### This Notebook demonstrates how to generate code from RAG docs.
we are using langchain python and js documentation as RAG.
We will use the following steps:
1. Load the documents
2. Create a vector store
3. Generate code from the documents
4. Save the generated code
5. Run the generated code
6. Display the output of the generated code


Step 0: setup (* installation in  requirements.txt and env examples for set env variables)

In [10]:
import os
from dotenv import load_dotenv

load_dotenv()
GoogleGeminiKey = os.getenv('GOOGLE_GEMINI_KEY')
connection = os.getenv("PG_CONNECTION_STRING")

Step 1: Create database

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
from langchain_community.document_loaders import WebBaseLoader
import bs4


def clean_text(text):
    return re.sub(r'[\u200b\u200e\u200f]', '', text).strip()


def extract_how_tos(link: str):
    response = requests.get(link)
    soup = BeautifulSoup(response.text, "html.parser")
    content_div = soup.find("div", class_="theme-doc-markdown")

    results = []
    current_module = None

    for tag in content_div.find_all(["h2", "h3", "ul"]):
        if tag.name in ["h2", "h3"]:
            current_module = {
                "module": clean_text(tag.get_text()),
                "how_tos": []
            }
            results.append(current_module)

        elif tag.name == "ul" and current_module:
            for li in tag.find_all("li"):
                a = li.find("a")
                if a and a.get("href"):
                    link = urljoin(BASE_URL, a["href"])
                    current_module["how_tos"].append({
                        "name": clean_text(a.get_text()),
                        "link": link
                    })

    return results


async def web_base_loader(page_urls):
    loader = WebBaseLoader(
        web_paths=page_urls,
        bs_kwargs={
            "parse_only": bs4.SoupStrainer(class_="theme-doc-markdown markdown"),
        },
        # bs_get_text_kwargs={"separator": " | ", "strip": True}
    )
    documents = []

    async for doc in loader.alazy_load():
        documents.append(doc)

    # assert len(documents) == 1, "No documents loaded from the web page"
    # print("Web page loaded successfully!")
    return documents

In [14]:
# for python docs:

BASE_URL = "https://python.langchain.com"
HOW_TO_URL = f"{BASE_URL}/docs/how_to/"

python_langchain_tutorial_links = extract_how_tos(HOW_TO_URL) 


In [66]:
# Summary chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai.chat_models import ChatGoogleGenerativeAI
from langchain_core.runnables import chain
from langchain_core.pydantic_v1 import BaseModel, Field


class PageDetails(BaseModel):
    "Document Web page summarization and meta data extraction."
    summary: str
    description: str
    tags: list[str]


summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are professional in summarizing documents.And having 20+ years of experience in Programming and development. So you can easily and efficiently summarize the given documentation of any module. You are also good at understanding and summarizing complex technical documents."),
    ("system", "Summary should not be long, and Also generate the short description of 50 words about the documentation and list of tags."),
    ("user",
     "Summarize the following documentation of Langchain for module {module}, document: {document}"),
])

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # Allows 1 request every 10 seconds
    check_every_n_seconds=0.1,  # Checks every 100 ms
    max_bucket_size=10  # Maximum burst size
)

llm_lite = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite", api_key=GoogleGeminiKey,
                                  temperature=0, rate_limiter=rate_limiter).with_structured_output(PageDetails)


def summary_generator(module, document):
    summary_chain = summary_prompt | llm_lite
    summary = summary_chain.invoke({"module": module, "document": document})
    return summary

In [ ]:
dc = python_langchain_tutorial_links[0]
doc = await web_base_loader([d['link'] for d in dc['how_tos']])

Fetching pages: 100%|##########| 2/2 [00:00<00:00,  9.84it/s]


In [58]:
summary_generator(dc['module'], doc[0])

PageDetails(summary='The LangChain ecosystem offers different packages for specific functionalities. The main package is installed with pip install langchain. Other packages include langchain-core, integration packages (e.g., langchain-openai), langchain-community, langchain-experimental, LangGraph, LangServe, LangChain CLI, and LangSmith SDK. Installation instructions are provided for each, including from source.', description='This document provides instructions on how to install various LangChain packages. It covers the main langchain package, ecosystem packages, integration packages, experimental packages, LangGraph, LangServe, LangChain CLI, and LangSmith SDK. It also explains how to install packages from source.', tags=['LangChain', 'installation', 'packages', 'pip', 'conda'])

In [ ]:
documents = []
summaries = []
for how_tos in python_langchain_tutorial_links:
    module = how_tos['module']
    print(f"Module: {module} , links : {len(how_tos['how_tos'])}")
    # for how_to in module['how_tos']:
    #     print(f"  - How To: {how_to['name']}")
    #     print(f"    Link: {how_to['link']}")
    links_in_module = [d['link'] for d in how_tos['how_tos']]
    titles = [d['name'] for d in how_tos['how_tos']]
    
    if(len(links_in_module) > 0):
        docs = await web_base_loader(links_in_module)
        for index,doc in enumerate(docs):
            output = summary_generator(module, doc)
            summaries.append(output.summary)
            doc.metadata['description'] = output.description
            doc.metadata['tags'] = output.tags
            doc.metadata['title'] = titles[index]
            doc.metadata['module'] = module
            documents.append(doc)
    else:
        print(f"No links found for module: {module}")

Module: Components , links : 0
No links found for module: Components
Module: Chat models , links : 17


Fetching pages: 100%|##########| 17/17 [00:02<00:00,  7.36it/s]


Module: Messages , links : 3


Fetching pages: 100%|##########| 3/3 [00:00<00:00,  4.84it/s]


Module: Prompt templates , links : 5


...

Fetching pages: 100%|##########| 3/3 [00:00<00:00,  5.12it/s]


Module: LangChain Expression Language (LCEL) , links : 13


Fetching pages: 100%|##########| 13/13 [00:01<00:00,  8.09it/s]


Module: LangGraph , links : 0
No links found for module: LangGraph
Module: LangSmith , links : 0
No links found for module: LangSmith
Module: Evaluation , links : 0
No links found for module: Evaluation
Module: Tracing , links : 2


Fetching pages: 100%|##########| 2/2 [00:00<00:00,  5.84it/s]


In [75]:
len(summaries),len(documents),documents[0].metadata 

(161,
 161,
 {'source': 'https://python.langchain.com/docs/how_to/installation/',
  'description': 'This document provides instructions on how to install various LangChain packages, including the main langchain package, integration packages, experimental packages, and other related tools like LangGraph, LangServe, LangChain CLI, and LangSmith SDK. It also covers installation from source.',
  'tags': ['LangChain', 'Installation', 'pip', 'conda', 'packages'],
  'title': 'How to: install LangChain packages',
  'module': 'Installation'})

### indexing and save to local and PGVector DB

In [78]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001", google_api_key=GoogleGeminiKey)

In [83]:
from langchain.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
import faiss

index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [ ]:
from uuid import uuid4
from langchain_core.documents import Document
id_key="doc_id"
doc_ids = [str(uuid4()) for _ in range(len(documents))]
summary_doc = [
    Document(page_content=s,metadata={id_key:doc_ids[i]}) for i,s in enumerate(summaries)
]
vector_store.add_documents(summary_doc)
vector_store.save_local("summaries_store_index")

['9bab3f49-056d-4b05-aa64-9f09c17540f0',
 'ad216864-1a81-480c-8d06-c5eac3ac6078',
 'fd7ce2a4-a89f-4274-a8b7-19378a61aa47',
 '18fc6be1-ff8f-4faf-a3b0-dc90d9f83ab9',
 'a079c708-e46b-4fc8-b975-0353212e6054',
 '82d73d47-dab5-40ee-b8f6-cd06e61e454b',
 'dbfa998b-e2f7-413e-8741-5fca7ff3cad4',
 '2489115d-869a-44df-a02f-99461344b0e1',
 '83dfcf37-03b3-4101-84c1-8e67d093c9aa',
 '30267cca-d18a-4971-b9e5-f147690682a4',
 '7c00471c-a4b1-463f-996e-13f28884aa06',
 'd5cfb03c-2e13-4806-82ef-acbcce51efc7',
 '58de586b-0ee1-46e3-8570-1147adfb1a23',
 'bf7ba9d1-a6a3-4c0b-a43d-7793604f8e55',
 'ae04a403-3035-4654-a7c4-7c565bbb71c7',
 '9ddca67a-51d1-4df9-8871-d4d24be674f2',
 'bd858ede-a417-42a1-a592-303c203a1803',
 '62d6b629-e6e6-4032-8555-f52415da43e1',
 'cc4b844a-b8bd-490b-b56e-33033bc339ce',
 '0972526f-643b-4a77-a0de-9095ccd483d7',
 '6f20e7f6-3c47-440b-a883-f3d352392977',
 '71e823ae-9557-4cbb-b9c6-e8ea36bd4cd0',
 '7d2a9309-9639-4ba1-80ab-4a0209c49209',
 'daa206c4-d90e-4f9b-b108-ed921dfce549',
 'ba5d254c-b019-

In [93]:
from langchain.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
import faiss

index2 = faiss.IndexFlatL2(len(embeddings.embed_query("document_index")))

vector_store_document = FAISS(
    embedding_function=embeddings,
    index=index2,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)
vector_store_document.add_documents(documents=documents,ids=doc_ids)
vector_store_document.save_local("documents_store_index")



In [94]:
python_how_to_doc_collection_name = "LangChain_Python_how_tos"
js_how_to_doc_collection_name = "LangChain_JavaScript_how_tos"


In [97]:
## PGVector 
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryStore
from langchain_postgres.vectorstores import PGVector

# the vector store to use to index the child chunks
vector_store_py = PGVector(
    embeddings=embeddings,
    collection_name=python_how_to_doc_collection_name,
    connection=connection,
    use_jsonb=True,
)

store_py = InMemoryStore()

retriever_py = MultiVectorRetriever( 
    vectorstore=vector_store_py,
    docstore=store_py,
    id_key=id_key,
)


# add the document summaries to the vector store for similarity search:
retriever_py.vectorstore.add_documents(summary_doc)

# Store the original document in the store, linked to the summaries via doc_id
# this allows us to first search summaries efficiently, then fetch the full docs when needed
retriever_py.docstore.mset(list(zip(doc_ids,documents)))

#### same for JS

In [102]:
BASE_URL = "https://js.langchain.com"
HOW_TO_URL = f"{BASE_URL}/docs/how_to/"

js_langchain_tutorial_links = extract_how_tos(HOW_TO_URL) 

documents_js = []
summaries_js = []
for how_tos in js_langchain_tutorial_links:
    module = how_tos['module']
    print(f"Module: {module} , links : {len(how_tos['how_tos'])}")
    # for how_to in module['how_tos']:
    #     print(f"  - How To: {how_to['name']}")
    #     print(f"    Link: {how_to['link']}")
    links_in_module = [d['link'] for d in how_tos['how_tos']]
    titles = [d['name'] for d in how_tos['how_tos']]
    
    if(len(links_in_module) > 0):
        docs = await web_base_loader(links_in_module)
        for index,doc in enumerate(docs):
            output = summary_generator(module, doc)
            summaries_js.append(output.summary)
            doc.metadata['description'] = output.description
            doc.metadata['tags'] = output.tags
            doc.metadata['title'] = titles[index]
            doc.metadata['module'] = module
            documents_js.append(doc)
    else:
        print(f"No links found for module: {module}")

Module: Installation , links : 1


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  4.84it/s]


Module: Key features , links : 4


Fetching pages: 100%|##########| 4/4 [00:00<00:00,  8.52it/s]


Module: LangChain Expression Language (LCEL) , links : 11


...

Fetching pages: 100%|##########| 3/3 [00:00<00:00,  5.55it/s]


Module: Q&A over graph databases , links : 4


Fetching pages: 100%|##########| 4/4 [00:00<00:00,  5.47it/s]


Module: LangGraph.js , links : 0
No links found for module: LangGraph.js
Module: LangSmith , links : 0
No links found for module: LangSmith
Module: Evaluation , links : 0
No links found for module: Evaluation
Module: Tracing , links : 2


Fetching pages: 100%|##########| 2/2 [00:00<00:00,  5.45it/s]


In [104]:
from langchain.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
import faiss
from uuid import uuid4
from langchain_core.documents import Document

index_js = faiss.IndexFlatL2(len(embeddings.embed_query("js_langchain_summary_store")))

vector_store_js = FAISS(
    embedding_function=embeddings,
    index=index_js,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

id_key="doc_id"
doc_ids_js = [str(uuid4()) for _ in range(len(documents_js))]
summary_doc_js = [
    Document(page_content=s,metadata={id_key:doc_ids_js[i]}) for i,s in enumerate(summaries_js)
]
vector_store_js.add_documents(summary_doc_js)
vector_store_js.save_local("summaries_store_index_js")

index_docs_js = faiss.IndexFlatL2(len(embeddings.embed_query("js_langchain_document_store")))

vector_store_doc_js = FAISS(
    embedding_function=embeddings,
    index=index_docs_js,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

vector_store_doc_js.add_documents(documents=documents_js,ids=doc_ids_js)
vector_store_doc_js.save_local("document_store_index_js")

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


RuntimeError: Error in void __cdecl faiss::write_index(const Index *, IOWriter *, int) at D:\a\faiss-wheels\faiss-wheels\faiss\faiss\impl\index_write.cpp:412: Error: 'ret == (size * 4)' failed: write error in summaries_store_index_js\index.faiss: 4051 != 408576 (No space left on device)

In [ ]:
## PGVector 
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryStore
from langchain_postgres.vectorstores import PGVector

# the vector store to use to index the child chunks
vector_store_js = PGVector(
    embeddings=embeddings,
    collection_name=js_how_to_doc_collection_name,
    connection=connection,
    use_jsonb=True,
)

store_js = InMemoryStore()

retriever_js = MultiVectorRetriever( 
    vectorstore=vector_store_js,
    docstore=store_js,
    id_key=id_key,
)


# add the document summaries to the vector store for similarity search:
retriever_js.vectorstore.add_documents(summary_doc_js)

# Store the original document in the store, linked to the summaries via doc_id
# this allows us to first search summaries efficiently, then fetch the full docs when needed
retriever_js.docstore.mset(list(zip(doc_ids_js,documents_js)))

In [ ]:
from typing import Literal
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_google_genai.chat_models import RecursiveCharacterTextSplitter

class QueryRoute(BaseModel):
    "Route a user query to the most relevant datasource"
    document